# AI-SOC Full Pipeline Demonstration

In [1]:
import pandas as pd
import numpy as np
import joblib
import random

In [2]:
binary_model = joblib.load("../models/binary_model.pkl")

multiclass_model = joblib.load("../models/multiclass_model.pkl")

scaler = joblib.load("../models/scaler.pkl")

print("Models and preprocessing artifacts loaded successfully.")

Models and preprocessing artifacts loaded successfully.


In [3]:
df = pd.read_csv("../data/processed/unsw_nb15_processed.csv")

sample = df.sample(1, random_state=42).copy()

print("Incoming network traffic detected")

sample.head()

Incoming network traffic detected


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,label
6252,-0.292481,-1.400577,0.39307,-0.382936,-0.109469,0.024845,-0.179994,-0.081937,-0.071408,-0.420855,...,-0.842689,-0.120294,-0.736066,-0.542306,-0.7187,-0.431437,-0.420724,-0.518969,0.315927,0


In [5]:
X = sample.drop(columns=["label"], errors="ignore")

expected_cols = scaler.feature_names_in_

for col in expected_cols:
    if col not in X.columns:
        X[col] = 0

X = X[expected_cols]

X_scaled = scaler.transform(X)

print("Traffic preprocessed successfully")

Traffic preprocessed successfully


In [6]:
binary_prediction = binary_model.predict(X_scaled)[0]
binary_proba = binary_model.predict_proba(X_scaled)[0][1]

if binary_prediction == 1:
    detection_result = "ATTACK"
else:
    detection_result = "NORMAL"

print(f"Detection Result: {detection_result}")
print(f"Model Confidence: {round(binary_proba*100,2)}%")


Detection Result: NORMAL
Model Confidence: 1.0%


/home/uzxn/AI-SOC/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/uzxn/AI-SOC/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [7]:
if detection_result == "ATTACK":
    attack_type = multiclass_model.predict(X_scaled)[0]
    print(f"⚔ Attack Type Identified: {attack_type}")
else:
    attack_type = "N/A"


In [9]:
malicious_ips = {
    "175.45.176.3": 90,
    "149.171.126.14": 75,
    "59.166.0.0": 60
}

if "srcip" in sample.columns:
    source_ip = str(sample["srcip"].iloc[0])
else:
    source_ip = "149.171.126.14"

reputation_score = malicious_ips.get(source_ip, random.randint(10,50))

print(f"Threat Intelligence Reputation Score: {reputation_score}")

Threat Intelligence Reputation Score: 45


In [10]:
risk_score = (binary_proba * 0.6) + (reputation_score/100 * 0.4)

if risk_score >= 0.85:
    risk_level = "CRITICAL"
elif risk_score >= 0.65:
    risk_level = "HIGH"
elif risk_score >= 0.4:
    risk_level = "MEDIUM"
else:
    risk_level = "LOW"

print(f"Calculated Risk Score: {round(risk_score,2)}")
print(f"Risk Level: {risk_level}")

Calculated Risk Score: 0.19
Risk Level: LOW


In [11]:
if risk_level == "CRITICAL":
    action = "BLOCK IP + ISOLATE HOST"
elif risk_level == "HIGH":
    action = "BLOCK IP"
elif risk_level == "MEDIUM":
    action = "ALERT SOC ANALYST"
else:
    action = "MONITOR TRAFFIC"

print(f"Automated Response: {action}")

Automated Response: MONITOR TRAFFIC


In [12]:
print("\n==============================")
print("AI-SOC ALERT REPORT")
print("==============================")
print(f"Source IP: {source_ip}")
print(f"Detection: {detection_result}")
print(f"Attack Type: {attack_type}")
print(f"Model Confidence: {round(binary_proba*100,2)}%")
print(f"Reputation Score: {reputation_score}")
print(f"Risk Level: {risk_level}")
print(f"Recommended Action: {action}")
print("==============================")



AI-SOC ALERT REPORT
Source IP: -0.2924814152069858
Detection: NORMAL
Attack Type: N/A
Model Confidence: 1.0%
Reputation Score: 45
Risk Level: LOW
Recommended Action: MONITOR TRAFFIC
